# Learning costs (structured perceptron)

_Artificial Intelligence — more_

**Score whole structures. When you pick the wrong one, reward the right one and punish your pick.**

Search needs costs (or scores) on actions. Where do those numbers come from? We can learn them.

     The structured perceptron scores a whole output (a path, a parse, a labeling) and fixes its weights when it picks the wrong one.

---

This notebook is a practice scaffold for the **Learning costs (structured perceptron)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np

# features = edge-usage counts over 4 edges [e1, e2, e3, e4].
# True path uses e1,e2 ; one impostor path uses e3,e4.
phi_true    = np.array([1, 1, 0, 0])
phi_wrong   = np.array([0, 0, 1, 1])

w = np.zeros(4)
for step in range(3):
    # predict the highest-scoring path; on a tie prefer the impostor (counts as a mistake)
    s_true, s_wrong = w @ phi_true, w @ phi_wrong
    phi_hat = phi_wrong if s_wrong >= s_true else phi_true
    if not np.array_equal(phi_hat, phi_true):     # mistake -> structured perceptron update
        w = w + phi_true - phi_hat
    print(f"step {step}: w={w.tolist()}  true={w@phi_true:.0f}  wrong={w@phi_wrong:.0f}")

print("final weights:", w.tolist())
print("true path wins?", (w @ phi_true) > (w @ phi_wrong))

## Visualize the data & results

_POS-tagging 'dogs chase cats': does one perceptron update make the correct NOUN VERB NOUN tagging win?_

In [ ]:
import matplotlib.pyplot as plt

# POS tagging the real sentence "dogs chase cats". Gold = NOUN VERB NOUN.
sent = ["dogs", "chase", "cats"]
gold  = ["NOUN", "VERB", "NOUN"]
wrong = ["NOUN", "NOUN", "NOUN"]

def feats(words, tags):
    f = {}
    for w, t in zip(words, tags):                 # emission features
        f[("emit", w, t)] = f.get(("emit", w, t), 0) + 1
    for i in range(1, len(tags)):                 # transition features
        k = ("trans", tags[i-1], tags[i])
        f[k] = f.get(k, 0) + 1
    return f

def score(w, f):
    return sum(w.get(k, 0.0) * v for k, v in f.items())

fg, fw = feats(sent, gold), feats(sent, wrong)
w = {}
before = [score(w, fg), score(w, fw)]             # both 0
for k, v in fg.items(): w[k] = w.get(k, 0.0) + v  # +gold features
for k, v in fw.items(): w[k] = w.get(k, 0.0) - v  # -predicted features
after = [score(w, fg), score(w, fw)]              # +3 vs -5

labels = ["correct: NOUN VERB NOUN", "wrong: NOUN NOUN NOUN"]
colors = ["#7ee787", "#ff7b72"]
fig, ax = plt.subplots()
bars = ax.bar(labels, after, color=colors)
for b, v in zip(bars, after):
    ax.text(b.get_x()+b.get_width()/2, v, str(int(v)),
            ha="center", va="bottom" if v >= 0 else "top")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("Score of tagging 'dogs chase cats' before vs after one update")
plt.show()